In [ ]:
%pip install yfinance ddgs
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
import crewai.memory.storage.kickoff_task_outputs_storage as kickoff_storage
import yfinance as yf
from ddgs import DDGS

# Force CrewAI kickoff SQLite storage to a writable folder in this project.
project_data_dir = Path.cwd() / ".crewai_data"
project_data_dir.mkdir(parents=True, exist_ok=True)
kickoff_storage.db_storage_path = lambda: str(project_data_dir)

# ==========================================
# 1. THE 100% FREE BRIDGE TO OLLAMA
# ==========================================
# This tells CrewAI to look at your local machine instead of OpenAI
local_llm = LLM(
    model="ollama/llama3.1:8b",  # Tool-capable for CrewAI agents
    base_url="http://localhost:11434"
)

# ==========================================
# 2. BUILDING THE TOOLS (Giving AI hands)
# ==========================================

@tool("Stock Price Fetcher")
def fetch_stock_price(ticker: str) -> str:
    """Fetches the last 30 days of closing prices for a given stock ticker."""
    print(f"\n[TOOL CALLED] Fetching 30-day price data for {ticker}...")
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1mo")
    return f"30-day price history for {ticker}:\n{hist['Close'].to_string()}"

@tool("News Scraper")
def fetch_stock_news(ticker: str) -> str:
    """Searches the web for the latest financial news about a given stock ticker."""
    print(f"\n[TOOL CALLED] Searching the web for {ticker} news...")
    query = f"latest financial news {ticker} stock"
    try:
        results = list(DDGS().text(query, max_results=3))
    except Exception as e:
        return f"Could not fetch news for {ticker}: {e}"

    if not results:
        return f"Latest news for {ticker}:\nNo recent results found."

    news_summary = "\n".join(
        [f"- {res.get('title', 'No title')}: {res.get('body', 'No summary')}" for res in results]
    )
    return f"Latest news for {ticker}:\n{news_summary}"

print("Environment setup complete! Tools and Local LLM are ready.")
print(f"CrewAI storage root: {project_data_dir}")

In [ ]:
# Test the tool with Tesla
tesla_news = fetch_stock_news.run("TSLA")
print(tesla_news)


In [ ]:
# ==========================================
# 3. DEFINING THE AGENTS (The "Who")
# ==========================================
quant = Agent(
    role="Senior Data Fetcher",
    goal="Accurately retrieve and summarize mathematical stock data for {ticker}.",
    backstory="You are a ruthless, numbers-driven Wall Street quant. You only care about the raw data and 30-day trends.",
    tools=[fetch_stock_price],
    llm=local_llm,
    verbose=True
)

analyst = Agent(
    role="News & Media Analyst",
    goal="Gauge public perception and find potential catalysts for {ticker} based on recent news.",
    backstory="You are an expert at reading between the lines of financial news. You identify if the media sentiment is bullish (positive) or bearish (negative).",
    tools=[fetch_stock_news],
    llm=local_llm,
    verbose=True
)

cio = Agent(
    role="Lead Portfolio Manager",
    goal="Synthesize data and news to write a final, structured investment memo for {ticker}.",
    backstory="You are a cautious but decisive hedge fund manager. You take the raw data from your Quant and the sentiment from your Analyst to make a final Buy, Hold, or Sell recommendation.",
    llm=local_llm,
    verbose=True
)

# ==========================================
# 4. DEFINING THE TASKS (The "What")
# ==========================================
task_price = Task(
    description="Use your tool to fetch the 30-day price history for {ticker}. Format it into a clean summary indicating if the trend is up, down, or flat.",
    expected_output="A short paragraph summarizing the 30-day price trend.",
    agent=quant
)

task_news = Task(
    description="Search the web for the latest news on {ticker}. Summarize the top events and declare the overall sentiment (Bullish, Bearish, or Neutral).",
    expected_output="A bulleted list of news summaries and a final sentiment verdict.",
    agent=analyst
)

task_memo = Task(
    description="Review the price data from the Quant and the news sentiment from the Analyst. Write a 3-paragraph investment memo concluding with a definitive Buy, Hold, or Sell rating for {ticker}.",
    expected_output="A 3-paragraph investment memo with a clear final recommendation.",
    agent=cio,
    context=[task_price, task_news]
)

# ==========================================
# 5. FORMING THE CREW AND RUNNING IT
# ==========================================
quant_crew = Crew(
    agents=[quant, analyst, cio],
    tasks=[task_price, task_news, task_memo],
    verbose=True
)

target_ticker = "AAPL"  # Change this to any ticker you want to analyze
print(f"\nKicking off research on {target_ticker}...\n")

# This single line passes the ticker to all agents and starts the sequence
try:
    result = quant_crew.kickoff(inputs={"ticker": target_ticker})
except Exception as e:
    print(f"Crew run failed: {e}")
    msg = str(e).lower()
    if "does not support tools" in msg:
        print("Your Ollama model does not support tools. Try: ollama pull llama3.1:8b")
    elif "connection" in msg or "refused" in msg:
        print("Could not reach Ollama. Start it first with: ollama serve")
else:
    print("\n==========================================")
    print("FINAL INVESTMENT MEMO:")
    print("==========================================")
    print(result)
